In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/murtazaabdullah2010/kz-task-1-llm-based-text/kz_test.csv
/kaggle/input/datasets/tamerlanomralinov/jimmy-and-aibek-brotherhood/train_data.csv
/kaggle/input/datasets/tamerlanomralinov/jimmy-and-aibek-brotherhood/test_data.csv
/kaggle/input/datasets/tamerlanomralinov/jimmy-and-aibek-brotherhood/sample_output.csv
/kaggle/input/datasets/tamerlanomralinov/jimmy-and-aibek-brotherhood/custom_archive.py


In [2]:
train_ds= pd.read_csv("/kaggle/input/datasets/tamerlanomralinov/jimmy-and-aibek-brotherhood/train_data.csv")
test_ds = pd.read_csv("/kaggle/input/datasets/tamerlanomralinov/jimmy-and-aibek-brotherhood/test_data.csv")
sample_sub = pd.read_csv("/kaggle/input/datasets/tamerlanomralinov/jimmy-and-aibek-brotherhood/sample_output.csv")

In [4]:
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel

In [8]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")
class DF(nn.Module):
    def __init__(self, df, tokenizer):
        self.df = df
        self.cpp = self.df["cpp_source"]
        self.py = self.df["py_source"]
        self.tok = tokenizer
    def __len__(self, ):
        return len(self.df)
    def __getitem__(self, idx):
        cpp = self.cpp[idx]
        py = self.py[idx]
        tok1= self.tok(py, max_length = 256, padding = "max_length", truncation = True, return_tensors = "pt")
        tok2 = self.tok(cpp, max_length= 256, padding = "max_length", truncation = True ,return_tensors = "pt")
        input_ids1, attn_mask1 = tok1["input_ids"].squeeze(0), tok1["attention_mask"].squeeze(0)
        input_ids2, attn_mask2 = tok2["input_ids"].squeeze(0), tok2["attention_mask"].squeeze(0)
        return input_ids1, attn_mask1, input_ids2, attn_mask2

train_df = DF(train_ds, tokenizer)
train_dl  = DataLoader(train_df, batch_size= 16, shuffle = True, num_workers = 4, pin_memory = True)
for batch in train_dl:
    x1, x2, y1, y2= batch
    print(x1.shape)
    print(y1.shape)
    break

torch.Size([16, 256])
torch.Size([16, 256])


In [48]:
class MODEL(nn.Module):
    def __init__(self):
        super().__init__()
        self.back = AutoModel.from_pretrained("microsoft/codebert-base")
        self.fc = nn.Linear(768, 256)
        self.fc= nn.Sequential(
            nn.Linear(768, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Linear(512, 512)
        )
    def forward(self, x1, x2):
        x = self.back(input_ids=x1, attention_mask=x2).last_hidden_state[:, 0, :]
        return self.fc(x)
device = ("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
model = MODEL().to(device)
opt =torch.optim.AdamW(model.parameters(), lr= 2e-5)
model.train()
temp = 0.07
for e in range(10):
    t_loss = 0
    for batch in train_dl:
        x1, x2, y1, y2 = batch
        x1, x2, y1, y2 = x1.to(device), x2.to(device), y1.to(device), y2.to(device)
        py = F.normalize(model(x1, x2), dim = 1)
        cpp = F.normalize(model(y1, y2), dim = 1)
        l = py @ cpp.T
        labels= torch.arange(l.size(0)).to(device)
        opt.zero_grad()
        loss= F.cross_entropy(logits/temp, labels)
        loss.backward()
        opt.step()
        t_loss  +=loss.item()
    print("E: ", e, "train_loss : ", t_loss/len(train_dl))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

E:  0 train_loss :  2.5684355156762257
E:  1 train_loss :  2.376840114593506
E:  2 train_loss :  2.061248404639108
E:  3 train_loss :  2.1400437355041504
E:  4 train_loss :  1.8373916149139404
E:  5 train_loss :  1.76853266784123
E:  6 train_loss :  1.718666442802974
E:  7 train_loss :  1.4996677892548698


In [39]:
cpp_embeds = []
model.eval()
with torch.no_grad():
    for idx, row in test_ds[200:].iterrows():
        code = row["source"]
        tok = tokenizer(code,max_length=256,padding="max_length", truncation=True,return_tensors="pt")
        input_ids = tok["input_ids"].to(device)
        attn_mask = tok["attention_mask"].to(device)
        emb = model(input_ids, attn_mask)  
        emb = F.normalize(emb, dim=1)
        cpp_embeds.append(emb.squeeze(0).cpu())

In [40]:
cpp_embeds = torch.stack(cpp_embeds) 

In [41]:
py_embeds = []

with torch.no_grad():
    for idx, row in test_ds[:200].iterrows():
        code = row["source"]
        tok = tokenizer(code,max_length=256,padding="max_length", truncation=True,return_tensors="pt")
        input_ids = tok["input_ids"].to(device)
        attn_mask = tok["attention_mask"].to(device)
        emb = model(input_ids, attn_mask)
        emb = F.normalize(emb, dim=1)
        py_embeds.append(emb.squeeze(0).cpu())

In [42]:
py_embeds = torch.stack(py_embeds)
sim_mat = py_embeds @ cpp_embeds.T 

In [43]:
results = []
cpp_ids = test_ds[200:]["datapointID"].values
for i in range(sim_mat.shape[0]):
    scores = sim_mat[i]
    top_indices = torch.topk(scores, k=20).indices.tolist()
    candidates = [cpp_ids[j] for j in top_indices]
    results.append(";".join(map(str, candidates)))

In [44]:
sample_sub["answer"] = results

In [45]:
sample_sub.to_csv("sub3.csv", index= False)